# v7c — Ensemble: TF-IDF + Semantic Embeddings

## Purpose
Combine our v6 best features (word+char TF-IDF, 6K sparse) with dense semantic
embeddings (384-dim from sentence-transformers). This tests whether contextual
meaning adds information beyond bag-of-words.

## Approaches
1. **Feature concatenation**: [6K TF-IDF | 384 embeddings] → single classifier
2. **Soft voting**: Weighted average of TF-IDF classifier + embedding classifier

## Expected runtime: 3-5 min (embedding step ~2 min on CPU)

## Kernel: efaai_v3 (Python 3.12)

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.path.join(ROOT, ".."))

CSV = os.path.join(ROOT, "data", "v5a_eos_vs_noneos.csv")
TEXT_COL = "PSI Failure Desc"
LABEL_COL = "label"
print(f"ROOT: {ROOT}")

ROOT: <project-root>


In [2]:
# Load and split (same as v5a/v6)
df = pd.read_csv(CSV)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 3].reset_index(drop=True)
le = LabelEncoder()
df["y"] = le.fit_transform(df[LABEL_COL])
EOS_IDX = list(le.classes_).index("EOS")
X_text = df[TEXT_COL].values
y = df["y"].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(y_train)}, Test: {len(y_test)}, EOS%: {(y==EOS_IDX).mean()*100:.1f}%")

Train: 11128, Test: 2782, EOS%: 25.5%


In [3]:
# TF-IDF features (v6 best)
print("Building TF-IDF features...")
tw = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=3000, sublinear_tf=True)
tc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=3000, sublinear_tf=True)

X_train_tfidf = hstack([tw.fit_transform(X_train_text), tc.fit_transform(X_train_text)])
X_test_tfidf = hstack([tw.transform(X_test_text), tc.transform(X_test_text)])
print(f"TF-IDF shape: {X_train_tfidf.shape}")

Building TF-IDF features...


TF-IDF shape: (11128, 6000)


## §3 — Semantic Embeddings

Using ll-MiniLM-L6-v2 (384-dim, fast, good quality).
This captures meaning that TF-IDF misses: synonyms, paraphrases, context.

In [4]:
# Semantic embeddings
from sentence_transformers import SentenceTransformer

print("Loading sentence-transformer model...")
model_st = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding train texts...")
t0 = time.time()
X_train_emb = model_st.encode(X_train_text.tolist(), show_progress_bar=True, batch_size=64)
print(f"  Train embeddings: {X_train_emb.shape} ({time.time()-t0:.1f}s)")

print("Encoding test texts...")
X_test_emb = model_st.encode(X_test_text.tolist(), show_progress_bar=True, batch_size=64)
print(f"  Test embeddings: {X_test_emb.shape}")

Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding train texts...


Batches:   0%|          | 0/174 [00:00<?, ?it/s]

  Train embeddings: (11128, 384) (55.5s)
Encoding test texts...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

  Test embeddings: (2782, 384)


## §4 — Approach 1: Feature Concatenation

Combine [6K TF-IDF | 384 embeddings] = 6384-dim feature vector.

In [5]:
# Concatenate TF-IDF + embeddings
X_train_concat = hstack([X_train_tfidf, csr_matrix(X_train_emb)])
X_test_concat = hstack([X_test_tfidf, csr_matrix(X_test_emb)])
print(f"Concatenated shape: {X_train_concat.shape}")

def evaluate(y_true, y_pred, label):
    mf1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    ef1 = f1_score(y_true, y_pred, pos_label=EOS_IDX)
    print(f"  {label}: Macro-F1={mf1:.4f}, Acc={acc:.4f}, EOS-F1={ef1:.4f}")
    return {"model": label, "macro_f1": mf1, "accuracy": acc, "eos_f1": ef1}

results = []

# Baseline: TF-IDF only + SMOTE + RF
print("Baseline: TF-IDF only + SMOTE + RF")
sm = SMOTE(random_state=42)
Xr, yr = sm.fit_resample(X_train_tfidf, y_train)
rf_base = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_base.fit(Xr, yr)
results.append(evaluate(y_test, rf_base.predict(X_test_tfidf), "TF-IDF only (baseline)"))

# Concat + SMOTE + RF
print("\nConcat (TF-IDF+Emb) + SMOTE + RF")
Xr2, yr2 = sm.fit_resample(X_train_concat, y_train)
rf_concat = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_concat.fit(Xr2, yr2)
results.append(evaluate(y_test, rf_concat.predict(X_test_concat), "Concat + SMOTE + RF"))

# Concat + BalancedRF
print("\nConcat + BalancedRF")
brf = BalancedRandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
brf.fit(X_train_concat, y_train)
results.append(evaluate(y_test, brf.predict(X_test_concat), "Concat + BalancedRF"))

Concatenated shape: (11128, 6384)
Baseline: TF-IDF only + SMOTE + RF


  TF-IDF only (baseline): Macro-F1=0.7725, Acc=0.8339, EOS-F1=0.6542

Concat (TF-IDF+Emb) + SMOTE + RF


  Concat + SMOTE + RF: Macro-F1=0.7602, Acc=0.8285, EOS-F1=0.6322

Concat + BalancedRF


  Concat + BalancedRF: Macro-F1=0.7584, Acc=0.8217, EOS-F1=0.6348


## §5 — Approach 2: Soft Voting

Train separate classifiers on each feature set, average their probabilities
with different weights.

In [6]:
# Soft voting with weight sweep
print("Soft Voting (TF-IDF clf + Embedding clf):")

# Train embedding-only classifier
sm3 = SMOTE(random_state=42)
Xr3, yr3 = sm3.fit_resample(X_train_emb, y_train)
rf_emb = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_emb.fit(Xr3, yr3)

# Get probabilities
prob_tfidf = rf_base.predict_proba(X_test_tfidf)
prob_emb = rf_emb.predict_proba(X_test_emb)

best_w, best_mf1 = 0, 0
for w_tfidf in [0.5, 0.6, 0.7, 0.8]:
    w_emb = 1 - w_tfidf
    prob_ensemble = w_tfidf * prob_tfidf + w_emb * prob_emb
    preds_ens = prob_ensemble.argmax(axis=1)
    mf1 = f1_score(y_test, preds_ens, average="macro")
    acc = accuracy_score(y_test, preds_ens)
    ef1 = f1_score(y_test, preds_ens, pos_label=EOS_IDX)
    print(f"  w_tfidf={w_tfidf:.1f}: Macro-F1={mf1:.4f}, Acc={acc:.4f}, EOS-F1={ef1:.4f}")
    if mf1 > best_mf1:
        best_mf1, best_w = mf1, w_tfidf

print(f"\n  Best weight: w_tfidf={best_w:.1f}, Macro-F1={best_mf1:.4f}")
results.append({"model": f"Soft Voting (w_tfidf={best_w})", "macro_f1": best_mf1, "accuracy": acc, "eos_f1": ef1})

Soft Voting (TF-IDF clf + Embedding clf):


  w_tfidf=0.5: Macro-F1=0.7701, Acc=0.8339, EOS-F1=0.6489
  w_tfidf=0.6: Macro-F1=0.7700, Acc=0.8336, EOS-F1=0.6490
  w_tfidf=0.7: Macro-F1=0.7730, Acc=0.8354, EOS-F1=0.6541
  w_tfidf=0.8: Macro-F1=0.7731, Acc=0.8350, EOS-F1=0.6546

  Best weight: w_tfidf=0.8, Macro-F1=0.7731


In [7]:
# Summary
print("=" * 70)
print("  v7c ENSEMBLE EXPERIMENT — SUMMARY")
print("=" * 70)
rdf = pd.DataFrame(results)
rdf = rdf.sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(rdf.to_string(index=False))

baseline_mf1 = rdf[rdf["model"].str.contains("baseline")]["macro_f1"].values[0]
best = rdf.iloc[0]
delta = best["macro_f1"] - baseline_mf1
print(f"\nBest: {best['model']}")
print(f"Delta vs baseline: {delta:+.4f}")

if delta > 0.005:
    print("\u2192 Semantic embeddings ADD value beyond TF-IDF!")
else:
    print("\u2192 TF-IDF captures most signal; embeddings add minimal benefit.")
    print("   This is common when texts are short and keyword-driven.")

rdf.to_csv(os.path.join(ROOT, "results", "v7c_ensemble_results.csv"), index=False)
print("\nSaved: results/v7c_ensemble_results.csv")
print("\n\u2705 v7c complete.")

  v7c ENSEMBLE EXPERIMENT — SUMMARY
                    model  macro_f1  accuracy   eos_f1
Soft Voting (w_tfidf=0.8)  0.773123  0.835011 0.654628
   TF-IDF only (baseline)  0.772460  0.833932 0.654192
      Concat + SMOTE + RF  0.760220  0.828541 0.632228
      Concat + BalancedRF  0.758415  0.821711 0.634757

Best: Soft Voting (w_tfidf=0.8)
Delta vs baseline: +0.0007
→ TF-IDF captures most signal; embeddings add minimal benefit.
   This is common when texts are short and keyword-driven.



Saved: results/v7c_ensemble_results.csv

✅ v7c complete.
